In [18]:
import hashlib
import os
import platform
import uuid

# 修改为自己的学号
STUDENT_ID = 20244080332

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]      {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] conda1
[学号]      20244080332
[机器指纹] 59a04118ce1009ce
[RUN_ID]   20244080332-59a04118ce1009ce


In [19]:
import re

# 1. 检查字符串是否仅包含英文字母a-z、A-Z和数字0-9
def check_alnum(s):
    return bool(re.fullmatch(r'[a-zA-Z0-9]+', s))

# 2. 匹配一个a后面跟着零个或多个b的字符串
def match_a_zero_b(s):
    return bool(re.fullmatch(r'ab*', s))

# 3. 匹配一个a后面跟着一个或多个b的字符串
def match_a_one_b(s):
    return bool(re.fullmatch(r'ab+', s))

# 4. 匹配以a开头、以b结尾，且中间可以有任意字符的字符串
def match_a_start_b(s):
    return bool(re.fullmatch(r'a.*b', s))

# 5. 删除IP地址中各段数字的前导零，全0保留一个0
def clean_ip(ip_str):
    def sub_zero(m):
        seg = m.group()
        return '0' if all(c == '0' for c in seg) else seg.lstrip('0')
    return re.sub(r'\d+', sub_zero, ip_str)

# 6. 将字符串中的空格替换为下划线，并将下划线替换为空格
def swap_space_underline(s):
    s1 = re.sub(r' ', '\x01', s)
    s2 = re.sub(r'_', ' ', s1)
    s3 = re.sub(r'\x01', '_', s2)
    return s3

# 7. 将 yyyy-mm-dd 格式的日期转换为 dd-mm-yyyy 格式
def convert_date(s):
    return re.sub(r'(\d{4})-(\d{2})-(\d{2})', r'\3-\2-\1', s)

# 8. 找出给定字符串中所有以 a 或 e 开头的单词
def find_a_e_start_word(s):
    return re.findall(r'\b[aeAE]\w*', s)

# 9. 将给定字符串中的 Road 缩写为 Rd
def replace_road(s):
    return re.sub(r'\bRoad\b', 'Rd', s)

# 10. 找出字符串中所有长度至少为4个字符的单词
def find_word_ge4(s):
    return re.findall(r'\b\w{4,}\b', s)

# 11. 删除字符串中除字母和数字之外的所有字符
def keep_alnum(s):
    return re.sub(r'[^a-zA-Z0-9]', '', s)

# 12. 删除字符串中的括号及括号内的内容
def remove_bracket_content(s):
    return re.sub(r'\s*\(.*?\)', '', s)


#测试用例
if __name__ == "__main__":
    print("1.", check_alnum("Abc123"), check_alnum("Abc!123"))
    print("2.", match_a_zero_b("a"), match_a_zero_b("abbb"))
    print("3.", match_a_one_b("ab"), match_a_one_b("a"))
    print("4.", match_a_start_b("a123b"), match_a_start_b("a123c"))
    print("5.", clean_ip("192.168.001.010"), clean_ip("000.010.001.000"))
    print("6.", swap_space_underline("hi test_abc"))
    print("7.", convert_date("2026-11-26"))
    print("8.", find_a_e_start_word("apple egg cat Eye ant"))
    print("9.", replace_road("Green Road, Red Road"))
    print("10.", find_word_ge4("hi hello test a bc"))
    print("11.", keep_alnum("a!b@1#c$2"))
    print("12.")
    test_list = ["example (.com)", "w3resource", "github (.com)", "stackoverflow (.com)"]
    for item in test_list:
        print(remove_bracket_content(item))


1. True False
2. True True
3. True False
4. True False
5. 192.168.1.10 0.10.1.0
6. hi_test abc
7. 26-11-2026
8. ['apple', 'egg', 'Eye', 'ant']
9. Green Rd, Red Rd
10. ['hello', 'test']
11. ab1c2
12.
example
w3resource
github
stackoverflow


In [ ]:
!pip install requests beautifulsoup4

In [26]:
import urllib.request
import re
import pandas as pd
from bs4 import BeautifulSoup
pd.set_option('display.unicode.east_asian_width', True)
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)

def _clean(text):
    """清洗文本：去除首尾空白，合并中间连续空白"""
    if text is None:
        return ""
    return re.sub(r"\s+", " ", text).strip()

def _fetch_html(url):
    """读取本地文件或网络URL"""
    if not url.lower().startswith(("http://", "https://")):
        try:
            with open(url, "r", encoding="utf-8") as f:
                return f.read()
        except FileNotFoundError:
            print(f"错误: 找不到文件 {url}")
            return None

    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            charset = resp.headers.get_content_charset() or 'utf-8'
            return resp.read().decode(charset, errors='ignore')
    except Exception as e:
        print(f"网络请求失败: {e}")
        raise
#2. retrieve_climate_table
def retrieve_climate_table(s):
    """
    作业要求函数：获取气候表格对象
    """
    html = _fetch_html(s)
    if not html:
        return None

    soup = BeautifulSoup(html, 'html.parser')
    tables = soup.find_all('table')

    best_table = None
    max_score = -1

    for table in tables:
        score = 0
        text_content = table.get_text()

        keywords = ['1月', '2月', '3月', 'Jan', 'Feb', 'Mar', 'Temperature', 'Precipitation', '降水', '气温']
        for kw in keywords:
            if kw in text_content:
                score += 1
        if table.find('caption'):
            score += 5

        if score > max_score:
            max_score = score
            best_table = table
    if max_score <= 0 and tables:
        return tables[0]

    return best_table
#3. list_climate_table_row_names
def list_climate_table_row_names(s):
    """
    作业要求函数：提取行名列表
    """
    table = retrieve_climate_table(s)
    if not table:
        return None

    row_names = []
    rows = table.find_all('tr')
    for row in rows:
        # 尝试获取 th (表头行) 或 td (数据行)
        cells = row.find_all(['th', 'td'])
        if not cells:
            continue
        first_cell_text = _clean(cells[0].get_text())
        if first_cell_text and not first_cell_text.isdigit():
            # 避免重复添加表头（如 "Climate data for..."）
            if len(first_cell_text) > 2:
                row_names.append(first_cell_text)

    return row_names

def parse_table_to_dataframe(table_tag):
    """
    核心解析逻辑：将 HTML Table 转换为 Pandas DataFrame
    包含自动转置逻辑以应对不同网站的排版差异
    """
    if not table_tag:
        return pd.DataFrame()

    data = []
    headers = []
    rows = table_tag.find_all('tr')
    for row in rows:
        cells = row.find_all(['th', 'td'])
        row_data = [_clean(cell.get_text()) for cell in cells]
        if any(row_data):
            data.append(row_data)

    if not data:
        return pd.DataFrame()
    headers = data[0]
    body_data = data[1:]
    first_col_sample = [row[0] for row in body_data if row]
    is_vertical = any("月" in str(x) or x in ["Jan", "Feb", "Mar"] for x in first_col_sample[:3])
    if is_vertical:
        new_headers = ["指标"] + [row[0] for row in body_data]
        new_body = []
        for i in range(1, len(headers)):
            new_row = [headers[i]]
            for row in body_data:
                # 防止索引越界
                val = row[i] if i < len(row) else ""
                new_row.append(val)
            new_body.append(new_row)
        df = pd.DataFrame(new_body, columns=new_headers)
    else:
        min_len = min(len(headers), min(len(row) for row in body_data)) if body_data else len(headers)
        df = pd.DataFrame(body_data, columns=headers[:min_len])
    if not df.empty:
        df = df.set_index(df.columns[0])
    return df

if __name__ == "__main__":
    # 测试 URL
    test_url = "haikou.html"

    print("--- 测试 retrieve_climate_table ---")
    table_obj = retrieve_climate_table(test_url)
    if table_obj:
        print(f"成功获取到表格对象，类型为: {type(table_obj)}")
        print("\n--- 测试 list_climate_table_row_names ---")
        names = list_climate_table_row_names(test_url)
        print(f"提取到的行名列表:\n{names}")
        print("\n--- 最终数据结果 (DataFrame) ---")
        df = parse_table_to_dataframe(table_obj)
        print(df)
    else:
        print("未找到有效的气候表格。")

--- 测试 retrieve_climate_table ---
成功获取到表格对象，类型为: <class 'bs4.element.Tag'>

--- 测试 list_climate_table_row_names ---
提取到的行名列表:
['平均最高温度（°C）', '平均最低温度（°C）', '平均降水量（mm）']

--- 最终数据结果 (DataFrame) ---
                     1月   2月   3月   4月    5月    6月    7月    8月    9月   10月  11月       12月
月份                                                                                                 
平均最高温度（°C）  21.8  22.9  26.3  30.7   32.3   33.7   33.5   32.6   30.7   29.1  26.2       22.4
平均最低温度（°C）  16.1  17.2  19.6  23.2   25.1   25.9   26.0   25.9   25.0   23.7  21.0       17.8
平均降水量（mm）    20.3  38.2  50.4  90.1  175.6  220.5  214.7  262.1  255.8  221.9  72.4  34.3 [71]
